# uv 패키징과 pytest 테스트 격리

로컬 패키지를 실제로 빌드·설치·가져오고, fixture와 monkeypatch로 반복 가능한 테스트를 만드는 과정을 확인합니다.

## 실행 환경 준비

Python 3.12와 고정된 uv·uv-build·pytest·pip 버전을 사용합니다. 07절의 모듈, 08절의 pytest, 09절의 데코레이터 표기를 연결합니다. 10절의 Pydantic은 필요하지 않습니다. 모든 파일은 임시 폴더에 만들고 실습 뒤 자동으로 제거됩니다.

In [1]:
# 교재와 같은 패키징·테스트 도구 버전을 설치합니다.
%pip install -q pytest==9.1.1 uv==0.12.11 uv-build==0.12.11 pip==26.2

Note: you may need to restart the kernel to use updated packages.

## 기본 실행: 가장 작은 패키지 만들기

`src/report_tools/__init__.py`에는 설치 뒤 `import report_tools`로 가져올 코드를 씁니다. `pyproject.toml`의 `[project]` 구역은 배포 이름과 버전을, `[build-system]` 구역은 소스를 휠로 바꿀 도구를 지정합니다. 배포 이름 `report-tools`와 import 이름 `report_tools`의 하이픈·밑줄 차이도 확인합니다.

In [2]:
# 임시 폴더에 패키지 소스와 빌드 설정을 준비합니다.
from pathlib import Path
from tempfile import TemporaryDirectory
import os
import subprocess
import sys

workspace = TemporaryDirectory()
root = Path(workspace.name)
package = root / "src" / "report_tools"
package.mkdir(parents=True)
(package / "__init__.py").write_text("""def make_title(name: str) -> str:
    return name.strip().title()
""")
(root / "pyproject.toml").write_text("""[project]
name = "report-tools"
version = "0.1.0"
[build-system]
requires = ["uv_build==0.12.11"]
build-backend = "uv_build"
""")
print(root.name, "패키지 소스 준비")

tmpw_hpy_x1 패키지 소스 준비


## 빌드·설치·가져오기

현재 커널의 Python으로 `uv build --wheel`을 호출하면 임시 프로젝트의 소스와 설정을 입력으로 받아 `.whl` 파일을 만듭니다. 이어서 새 가상환경을 만들고 그 휠을 설치합니다. 마지막으로 설치 환경의 새 Python 프로세스가 패키지를 가져와 함수를 호출합니다. 빌드, 설치, import는 서로 다른 단계입니다.

In [3]:
# 휠을 빌드하고 격리된 위치에 설치한 뒤 새 프로세스로 가져옵니다.
built = subprocess.run(
    [sys.executable, "-m", "uv", "build", "--wheel"],
    cwd=root, text=True, capture_output=True, check=True,
)
wheel = next((root / "dist").glob("*.whl"))
venv = root / ".venv"
subprocess.run([sys.executable, "-m", "uv", "venv",
                str(venv), "--python", sys.executable],
               check=True, capture_output=True)
env_python = venv / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
subprocess.run([sys.executable, "-m", "uv", "pip", "install",
                "--python", str(env_python), str(wheel)],
               check=True, capture_output=True)
imported = subprocess.run(
    [str(env_python), "-c",
     "import report_tools; print(report_tools.make_title('  agent log  '))"],
    text=True, capture_output=True, check=True, cwd=root,
)
print(wheel.name, "->", imported.stdout.strip())

report_tools-0.1.0-py3-none-any.whl -> Agent Log


## 결과 확인: 배포 산출물과 import

휠 파일이 존재하는지만으로 설치 성공을 알 수 없습니다. 새 Python 프로세스가 설치된 `make_title()`에 문자열을 입력하고 `Agent Log`를 반환받았는지 검사합니다. 이어서 입력만 `weekly note`로 바꾸어 같은 설치된 함수가 `Weekly Note`를 반환하는지 확인합니다.

In [4]:
# 빌드 파일과 설치 후 함수 결과를 함께 확인합니다.
print("휠 존재:", wheel.exists())
print("첫 입력:", imported.stdout.strip())
changed = subprocess.run(
    [str(env_python), "-c",
     "import report_tools; print(report_tools.make_title('  weekly note  '))"],
    text=True, capture_output=True, check=True, cwd=root,
)
print("바꾼 입력:", changed.stdout.strip())
assert wheel.exists()
assert imported.stdout.strip() == "Agent Log"
assert changed.stdout.strip() == "Weekly Note"

휠 존재: True
첫 입력: Agent Log
바꾼 입력: Weekly Note


## Before: 공유 상태 때문에 두 번째 테스트가 실패함

두 테스트가 같은 목록을 수정하면 실행 순서가 결과에 영향을 줍니다. pytest를 실제 실행하고 실패 코드와 요약을 관찰합니다.

In [5]:
# 공유 목록 때문에 두 번째 테스트가 실패하는 파일을 실행합니다.
bad_test = root / "test_shared.py"
bad_test.write_text("""items: list[str] = []
def test_add_alpha() -> None:
    items.append("alpha")
    assert items == ["alpha"]
def test_add_beta() -> None:
    items.append("beta")
    assert items == ["beta"]
""")
failed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", bad_test.name],
    cwd=root, text=True, capture_output=True,
)
print("종료 코드:", failed.returncode)
print(failed.stdout.strip().splitlines()[-1])
assert failed.returncode == 1

종료 코드: 1
1 failed, 1 passed in 0.01s


## After: fixture와 monkeypatch로 입력을 분리함

`@pytest.fixture`는 바로 아래 `items()`를 준비 함수로 등록합니다. 테스트 함수가 같은 이름의 `items` 매개변수를 적으면 pytest가 호출 직전에 fixture를 실행하고 반환된 빈 목록을 전달합니다. 각 테스트 호출이 새 목록을 받으므로 앞 테스트의 `append()` 결과가 이어지지 않습니다. `monkeypatch`는 뒤의 별도 예제에서 다룹니다.

In [6]:
# 같은 두 테스트를 유지하고 fixture로 준비 목록만 분리합니다.
good_test = root / "test_isolated.py"
good_test.write_text("""import pytest
@pytest.fixture
def items() -> list[str]:
    return []
def test_add_alpha(items: list[str]) -> None:
    items.append("alpha")
    assert items == ["alpha"]
def test_add_beta(items: list[str]) -> None:
    items.append("beta")
    assert items == ["beta"]
""")
passed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", good_test.name],
    cwd=root, text=True, capture_output=True, check=True,
)
print(passed.stdout.strip().splitlines()[-1])
assert "2 passed" in passed.stdout


2 passed in 0.00s


## 추가 확인: 함수 대체와 함수 복원

목록 공유 문제를 고친 두 테스트와 별도로 monkeypatch를 확인합니다. `setattr()`은 `status_service.fetch_status`를 테스트 동안 `fake_status`에 연결하고, `setenv()`는 첫 테스트 안에서 `APP_MODE`를 `test`로 지정합니다. 코드는 첫 테스트 안의 환경 변수 값과, 다음 테스트에서 원래 함수가 반환하는 `대기`를 직접 확인합니다. 환경 변수의 테스트 후 복원값은 이 출력에서 따로 검사하지 않습니다. 아래 서비스 모듈은 네트워크를 호출하지 않습니다.

In [7]:
# 대체할 함수와 그 함수를 호출하는 코드를 별도 모듈로 만듭니다.
service_source = """def fetch_status() -> str:
    return "대기"

def report_status() -> str:
    return "처리 상태: " + fetch_status()
"""
(root / "status_service.py").write_text(service_source, encoding="utf-8")
print("실습용 상태 모듈 준비 완료")


실습용 상태 모듈 준비 완료


In [8]:
# 함수 대체와 다음 테스트의 원래 함수 반환을 검사합니다.
mock_source = """import os
import pytest
import status_service

def test_fake_status(monkeypatch: pytest.MonkeyPatch) -> None:
    def fake_status() -> str:
        return "완료"
    monkeypatch.setattr(status_service, "fetch_status", fake_status)
    monkeypatch.setenv("APP_MODE", "test")
    assert status_service.report_status() == "처리 상태: 완료"
    assert os.environ["APP_MODE"] == "test"

def test_original_status() -> None:
    assert status_service.report_status() == "처리 상태: 대기"
"""
(root / "test_mock.py").write_text(mock_source, encoding="utf-8")
print("함수 대체와 복원 검사 두 개 준비 완료")


함수 대체와 복원 검사 두 개 준비 완료


In [9]:
# 목록 테스트와 별도 파일을 실행해 함수 대체·복원 결과를 확인합니다.
mocked = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "test_mock.py"],
    cwd=root, text=True, capture_output=True, check=True,
)
print(mocked.stdout.strip().splitlines()[-1])
assert "2 passed" in mocked.stdout
workspace.cleanup()


2 passed in 0.00s
